# バレーボール動画 AI分析 - Step 2〜3: 全区間分析 + 検証

このノートブックは、Gemini API を使用して動画と選手写真からラリー分析を行う実装です。

## Step 2〜3 の範囲
- ffmpeg 720p H.264 faststart 再エンコード
- 3分区間 + 20秒オーバーラップ分割
- 各区間を Gemini API に送信
- JSON スキーマ検証
- 絶対秒変換
- 重複排除

## まだ実装しないもの
- シート書き込み
- GAS 実装
- UI

## セル1: ライブラリインストール

In [ ]:
!pip install google-genai gdown gspread google-auth -q

## セル2: インポート・設定

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

import google.genai as genai

try:
    from google.colab import auth, userdata
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# ローカル実行用の秘密ファイル
LOCAL_SECRET_FILE = 'analysis/.local_secrets.json'


def load_local_secrets(file_path):
    """ローカルの秘密ファイルを読み込む"""
    candidates = [
        Path(file_path),
        Path('.local_secrets.json'),
        Path('analysis/.local_secrets.json'),
    ]

    for p in candidates:
        if p.exists():
            try:
                data = json.loads(p.read_text(encoding='utf-8'))
                if isinstance(data, dict):
                    print(f'ローカル秘密ファイルを読み込みました: {p}')
                    return data
            except Exception:
                pass

    return {}


def get_secret(key_name):
    """シークレット値を取得（Colab Secrets > 環境変数 > ローカルファイル）"""
    if IN_COLAB:
        try:
            value = userdata.get(key_name)
            if value is not None:
                return str(value).strip()
        except Exception:
            pass

    env_value = str(os.getenv(key_name, '')).strip()
    if env_value:
        return env_value

    file_value = LOCAL_SECRETS.get(key_name, '')
    return str(file_value).strip()


LOCAL_SECRETS = load_local_secrets(LOCAL_SECRET_FILE)

# Gemini API Key
GEMINI_API_KEY = get_secret('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError(
        'GEMINI_API_KEY が設定されていません。\n'
        'Colab の場合は Secrets に GEMINI_API_KEY を登録してください。\n'
        'ローカルの場合は .local_secrets.json に {"GEMINI_API_KEY": "your-key"} を設定してください。'
    )

# ============================================
# 試合情報入力（設計書§10 セル1）
# ============================================

# 試合情報（ユーザー入力）
MATCH_DATE = input("試合日（YYYY-MM-DD）: ").strip() or datetime.now().strftime('%Y-%m-%d')
OPPONENT = input("相手チーム名: ").strip() or "相手チーム"
GAME_NUMBER = input("ゲーム番号（1, 2, 3...）: ").strip() or "1"

# match_id生成: {date}_{opponent}_game{N}
MATCH_ID = f"{MATCH_DATE}_{OPPONENT}_game{GAME_NUMBER}"

# analysis_run_id生成: run_{yyyymmdd}_{HHmmss}
NOW = datetime.now()
ANALYSIS_RUN_ID = f"run_{NOW.strftime('%Y%m%d')}_{NOW.strftime('%H%M%S')}"

# prompt_version（ノートブック内定数）
PROMPT_VERSION = "v1.0"

print(f'\n=== 試合情報 ===')
print(f'match_id: {MATCH_ID}')
print(f'analysis_run_id: {ANALYSIS_RUN_ID}')
print(f'prompt_version: {PROMPT_VERSION}')

print('\n設定読み込み完了')

## セル3: Google Drive マウント

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive マウント完了')
else:
    print('ローカル実行モード: Drive マウントはスキップされます')

## セル4: Gemini クライアント初期化

## セル5: システムプロンプト・ユーザープロンプト定義

In [ ]:
# ============================================
# プロンプト定義（設計書§6.3, §6.4）
# ============================================

SYSTEM_PROMPT = """あなたはバレーボールの試合分析の専門家です。

## タスク
試合動画の一部（約3分間）を視聴し、全ラリーを検出・分析してください。

## チーム識別
- 画面手前側 = 「自チーム」、画面奥側 = 「相手」
- コートチェンジなし。ユニフォーム・背番号なし。コート位置のみで識別

## 選手識別
- ユーザープロンプトの選手写真を参照
- 服装ではなく体格・髪型・身長で識別。不明ならnull

## ラリーの定義
サーブのトスアップから得点確定まで。タイムアウト・休憩・選手交代は除外。

## 分析項目
- point_team（必須）: "自チーム" / "相手"
- deciding_team（必須）: ラリーを終わらせたプレーをしたチーム
- receive_grade: A/B/C/D/null（自チームサーブ側ならnull）
- receiver: 選手名/null
- play_type（必須）: サーブ/スパイク/フェイント/プッシュ/ブロック/ディグ/2段トス/フリーボール
- player: 選手名/null
- result_detail: アウト/ネット/シャット/タッチネット/ダブルコンタクト/サービスエース/タッチアウト/タッチイン/吸い込み/ポジション/オーバーネット/お見合い/その他/null
- attack_type: Aクイック/Bクイック/Cクイック/Aセミ/Bセミ/Cセミ/並行/オープン/2段トス/ブロードワイド/ブロードショート/null
- blocker_count: "0"/"1"/"2"/"3"/null
- zone_from/zone_to: 12ゾーン/null
- our_defense_type: "ブロック"/"ディグ"/null（相手攻撃で失点時のみ。不明なら"ディグ"）

## 確信度
confidence（全体）+ field_confidences（項目別12項目）。
0.8以上:明確 / 0.5〜0.79:おそらく / 0.5未満:推測（null推奨）

## タイムスタンプ
区間動画の先頭からの秒数。重複ラリーもそのまま出力。

## 出力形式
{
  "rallies": [{
    "rally_number": 整数,
    "rally_start_sec": 小数, "rally_end_sec": 小数,
    "confidence": 0.0-1.0,
    "point_team": "自チーム"/"相手",
    "deciding_team": "自チーム"/"相手",
    "receive_grade": .., "receiver": ..,
    "play_type": .., "player": ..,
    "result_detail": .., "attack_type": ..,
    "blocker_count": .., "zone_from": .., "zone_to": ..,
    "our_defense_type": .., "note": "..",
    "field_confidences": { 12項目 }
  }]
}

## 注意事項
- 不鮮明なら確信度を低く。推測よりnull
- タイムアウト・休憩はラリーとして検出しない
- 区間先頭/末尾でラリーが途中でもそのまま出力
"""


def build_user_prompt(match_info, start_sec, end_sec):
    """ユーザープロンプト構築（設計書§6.4）"""
    opponent = match_info.get("opponent", "相手")
    set_num = match_info.get("set", 1)
    serve_team = match_info.get("serve_team", "自チーム")
    rotation = match_info.get("rotation", 1)
    
    prompt = f"""## 選手識別
選手写真を参照してください。体格・髪型・身長で識別。不明ならnull。

## 試合情報
相手: {opponent} / セット: {set_num} / サーブ権: {serve_team} / ローテーション: {rotation}

## 区間情報
セット全体の {start_sec:.1f}秒〜{end_sec:.1f}秒 の区間。前区間と20秒オーバーラップ。

## 指示
全ラリーを分析し、JSON形式で回答してください。"""
    
    return prompt

In [ ]:
# Gemini クライアント初期化
client = genai.Client(api_key=GEMINI_API_KEY)

# モデル設定（設計書セクション4.4に準拠）
MODEL_NAME = "gemini-2.5-flash"
print(f'Gemini クライアント初期化完了: {MODEL_NAME}')

## セル7: ヘルパー関数定義

In [ ]:
# ============================================
# ヘルパー関数
# ============================================

def upload_video_to_gemini(video_path):
    """動画をGeminiにアップロード"""
    print(f'動画アップロード: {video_path}')
    video_file = client.files.upload(file=video_path)
    print(f'アップロード完了: {video_file.name}')
    return video_file


def delete_video_file(video_file):
    """動画ファイルを削除"""
    try:
        client.files.delete(name=video_file.name)
        print(f'動画削除完了: {video_file.name}')
    except Exception as e:
        print(f'動画削除失敗: {e}')


def load_player_photos(players_dir, player_names):
    """選手写真を読み込む"""
    photos = []
    for name in player_names:
        photo_path = Path(players_dir) / f"{name}.jpg"
        if not photo_path.exists():
            print(f'警告: 写真が見つかりません: {photo_path}')
            continue
        with open(photo_path, 'rb') as f:
            photo_data = f.read()
        photos.append({
            'name': name,
            'mime_type': 'image/jpeg',
            'data': photo_data
        })
    print(f'選手写真読み込み完了: {len(photos)}枚')
    return photos


def validate_and_normalize_rally(rally):
    """ラリーデータのスキーマ検証・正規化（設計書§6）"""
    errors = []
    validated = {}
    
    # 必須フィールド
    required_fields = ['rally_number', 'rally_start_sec', 'rally_end_sec', 'confidence', 'point_team', 'deciding_team', 'play_type']
    for field in required_fields:
        if field not in rally:
            errors.append(f'必須フィールド欠落: {field}')
            return None, errors
    
    # 値の正規化
    validated['rally_number'] = int(rally['rally_number'])
    validated['rally_start_sec'] = float(rally['rally_start_sec'])
    validated['rally_end_sec'] = float(rally['rally_end_sec'])
    validated['confidence'] = float(rally['confidence'])
    
    # point_team, deciding_teamの正規化
    validated['point_team'] = rally['point_team']
    validated['deciding_team'] = rally['deciding_team']
    
    # "相手"→実際の相手チーム名（後で置換）
    if validated['point_team'] == '相手':
        validated['point_team'] = '相手'  # 後で置換
    if validated['deciding_team'] == '相手':
        validated['deciding_team'] = '相手'  # 後で置換
    
    # オプションフィールド（null許容）
    optional_fields = ['receive_grade', 'receiver', 'player', 'result_detail', 'attack_type', 'blocker_count', 'zone_from', 'zone_to', 'our_defense_type', 'note']
    for field in optional_fields:
        value = rally.get(field)
        if value is None or value == 'null':
            validated[field] = ''
        else:
            validated[field] = str(value).strip()
    
    # field_confidencesの補完
    field_confidences = rally.get('field_confidences', {})
    confidence_fields = ['point_team', 'deciding_team', 'receive_grade', 'receiver', 'play_type', 'player', 'result_detail', 'attack_type', 'blocker_count', 'zone_from', 'zone_to', 'our_defense_type']
    validated['field_confidences'] = {}
    for field in confidence_fields:
        validated['field_confidences'][field] = float(field_confidences.get(field, 0.0))
    
    return validated, errors


def convert_to_absolute_seconds(rally, segment_start_sec):
    """相対秒を絶対秒に変換"""
    rally['absolute_start_sec'] = segment_start_sec + rally['rally_start_sec']
    rally['absolute_end_sec'] = segment_start_sec + rally['rally_end_sec']
    return rally


def remove_duplicates(rallies, threshold_sec=5.0):
    """重複排除（設計書§3.3）"""
    if not rallies:
        return []
    
    # 絶対秒で昇順ソート
    sorted_rallies = sorted(rallies, key=lambda x: x['absolute_start_sec'])
    
    deduped = []
    for rally in sorted_rallies:
        if not deduped:
            deduped.append(rally)
        else:
            # 前のラリーとの時間差をチェック
            prev = deduped[-1]
            time_diff = rally['absolute_start_sec'] - prev['absolute_end_sec']
            if time_diff < threshold_sec:
                # 重複とみなす（後の区間の結果を採用）
                print(f'重複検出: ラリー{rally["rally_number"]}（時間差: {time_diff:.1f}秒）')
                continue
            deduped.append(rally)
    
    return deduped

## セル6: Google Drive Folder ID確定・File ID取得・セット順決定

In [ ]:
# ============================================
# Google Drive Folder ID確定・File ID取得・セット順決定
# ============================================

# Folder ID（ユーザー入力またはデフォルト）
DRIVE_FOLDER_ID = input("Google Drive Folder ID（空白の場合はスキップ）: ").strip()

# 動画File ID（ユーザー入力またはデフォルト）
DRIVE_FILE_ID = input("動画File ID（空白の場合はサンプルを使用）: ").strip() or "1iz8llxRGksr-jYpnwcy0FPvKeRAVjwQM"

# セット順決定（ユーザー入力またはデフォルト）
SET_ORDER = input("セット順（例: 1,2,3 または 空白で単一セット）: ").strip()
if SET_ORDER:
    SET_ORDER = [int(s.strip()) for s in SET_ORDER.split(',')]
else:
    SET_ORDER = [1]  # デフォルトはセット1のみ

print(f'\n=== Drive 設定 ===')
print(f'Folder ID: {DRIVE_FOLDER_ID or "未設定"}')
print(f'File ID: {DRIVE_FILE_ID}')
print(f'セット順: {SET_ORDER}')

## セル5: Google Drive から動画をダウンロード

In [ ]:
import gdown

def download_video_from_drive(file_id, output_path):
    """Google Drive の共有リンクから動画をダウンロード（gdown使用）"""
    print(f'Google Drive から動画をダウンロード: {file_id}')
    
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, output_path, quiet=False)
    
    file_size = Path(output_path).stat().st_size
    print(f'ダウンロード完了: {output_path} ({file_size / 1024 / 1024:.2f} MB)')
    return output_path


# サンプル動画のファイルID
DRIVE_FILE_ID = "1iz8llxRGksr-jYpnwcy0FPvKeRAVjwQM"

# ダウンロード先パス
if IN_COLAB:
    DOWNLOAD_DIR = "/content"
else:
    DOWNLOAD_DIR = "analysis"

DOWNLOAD_PATH = f"{DOWNLOAD_DIR}/sample_video.mp4"

# 動画をダウンロード
download_video_from_drive(DRIVE_FILE_ID, DOWNLOAD_PATH)

# ============================================
# 動画処理設定
# ============================================

# 動画ファイル（Google Drive からダウンロードしたものを使用）
FULL_VIDEO_PATH = DOWNLOAD_PATH

# ffmpeg 720p再エンコード
if IN_COLAB:
    ENCODED_VIDEO_PATH = "/content/video_720p.mp4"
    TEMP_DIR = "/content"
else:
    ENCODED_VIDEO_PATH = "analysis/video_720p.mp4"
    TEMP_DIR = "analysis"

# ffmpeg 720p H.264 faststart 再エンコード
print(f'720p再エンコード開始: {FULL_VIDEO_PATH}')
import subprocess
cmd = [
    'ffmpeg', '-i', FULL_VIDEO_PATH,
    '-vf', 'scale=-2:720',
    '-c:v', 'libx264',
    '-movflags', '+faststart',
    '-c:a', 'aac',
    '-y',  # 上書き許可
    ENCODED_VIDEO_PATH
]
subprocess.run(cmd, check=True, capture_output=True)

from pathlib import Path
encoded_size = Path(ENCODED_VIDEO_PATH).stat().st_size
print(f'720p再エンコード完了: {ENCODED_VIDEO_PATH} ({encoded_size / 1024 / 1024:.2f} MB)')

# 動画長を取得
cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', ENCODED_VIDEO_PATH]
result = subprocess.run(cmd, capture_output=True, text=True, check=True)
video_duration = float(result.stdout.strip())
print(f'動画長: {video_duration:.1f}秒')

# 区間設定
SEGMENT_DURATION = 180  # 3分
OVERLAP_SEC = 20

# 区間分割
import math
num_segments = math.ceil((video_duration - SEGMENT_DURATION) / (SEGMENT_DURATION - OVERLAP_SEC)) + 1 if video_duration > SEGMENT_DURATION else 1
print(f'分割区間数: {num_segments}')

segments = []
for i in range(num_segments):
    start = i * (SEGMENT_DURATION - OVERLAP_SEC)
    end = min(start + SEGMENT_DURATION, video_duration)
    if start >= video_duration:
        break
    segment_path = f"{TEMP_DIR}/segment_{i:02d}.mp4"
    
    print(f'区間{i}: {start:.1f}s 〜 {end:.1f}s')
    cmd = [
        'ffmpeg', '-ss', str(start), '-to', str(end),
        '-i', ENCODED_VIDEO_PATH,
        '-c', 'copy',  # 再エンコード済みなのでcopyで高速
        '-y',
        segment_path
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    
    segments.append({
        'index': i,
        'start_sec': start,
        'end_sec': end,
        'path': segment_path
    })
    segment_size = Path(segment_path).stat().st_size
    print(f'  → {segment_path} ({segment_size / 1024 / 1024:.2f} MB)')

print(f'\n合計区間数: {len(segments)}')

# 選手写真ディレクトリ
if IN_COLAB:
    PLAYERS_DIR = "/content/players"
else:
    PLAYERS_DIR = "analysis/players"

# 選手リスト
PLAYER_NAMES = []

# 試合情報
MATCH_INFO = {
    "opponent": "レジン",
    "set": 1,
    "serve_team": "レジン",
    "rotation": 1
}

print(f'選手写真ディレクトリ: {PLAYERS_DIR}')
print(f'選手リスト: {PLAYER_NAMES}')

In [ ]:
## セル10: 全区間分析実行

import time

def call_gemini_with_video_and_photos(video_file, player_photos, system_prompt, user_prompt, max_retries=3):
    """Gemini API を呼び出し（動画+選手写真+プロンプト）。リトライ付き。"""
    for attempt in range(max_retries):
        try:
            print(f'Gemini API 呼び出し開始... (試行 {attempt + 1}/{max_retries})')
            
            # コンテンツを構築
            contents = [user_prompt]
            
            # 選手写真を追加
            for photo in player_photos:
                contents.append({
                    'inline_data': {
                        'mime_type': photo['mime_type'],
                        'data': photo['data']
                    }
                })
            
            # 動画を追加
            contents.append(video_file)
            
            # API 呼び出し（システムプロンプトを含める）
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config={
                    "response_mime_type": "application/json",
                    "max_output_tokens": 65536,  # 設計書8192だが切り詰め対策で65536
                    "temperature": 0.2,  # 設計書通り（Step 2-3で問題があれば0.5へ）
                    "system_instruction": system_prompt
                }
            )
            
            print('API 呼び出し完了')
            return response
            
        except Exception as e:
            print(f'API 呼び出しエラー: {e}')
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # 指数バックオフ
                print(f'{wait_time}秒待機してリトライ...')
                time.sleep(wait_time)
            else:
                raise


def analyze_all_segments(segments, player_photos, system_prompt, match_info):
    """全区間を分析"""
    all_rallies = []
    skipped_segments = 0
    
    for seg in segments:
        print(f'\n=== 区間 {seg["index"]} ({seg["start_sec"]:.1f}s 〜 {seg["end_sec"]:.1f}s) ===')
        
        # 動画アップロード
        try:
            video_file = upload_video_to_gemini(seg['path'])
        except Exception as e:
            print(f'区間{seg["index"]} アップロード失敗: {e}')
            skipped_segments += 1
            continue
        
        # ユーザープロンプト構築
        user_prompt = build_user_prompt(match_info, seg['start_sec'], seg['end_sec'])
        
        # API 呼び出し
        try:
            response = call_gemini_with_video_and_photos(video_file, player_photos, system_prompt, user_prompt)
        except Exception as e:
            print(f'区間{seg["index"]} API 呼び出し失敗: {e}')
            try:
                delete_video_file(video_file)
            except:
                pass
            skipped_segments += 1
            continue
        
        # JSON パース・検証・正規化
        try:
            data = json.loads(response.text)
            if 'rallies' not in data:
                print(f'区間{seg["index"]} ralliesキーなし')
                skipped_segments += 1
                try:
                    delete_video_file(video_file)
                except:
                    pass
                continue
            
            rallies = data['rallies']
            print(f'区間{seg["index"]}: {len(rallies)}ラリー検出')
            
            for rally in rallies:
                # スキーマ検証・正規化
                validated, errors = validate_and_normalize_rally(rally)
                if validated is None:
                    print(f'  ラリー{rally.get("rally_number", "?")} 区間エラー: {errors}')
                    continue
                
                if errors:
                    print(f'  ラリー{rally.get("rally_number", "?")}: {len(errors)}件の警告')
                
                # 絶対秒変換
                validated = convert_to_absolute_seconds(validated, seg['start_sec'])
                validated['segment_index'] = seg['index']
                
                all_rallies.append(validated)
            
        except json.JSONDecodeError as e:
            print(f'区間{seg["index"]} JSON パースエラー: {e}')
            skipped_segments += 1
        
        # 動画削除
        try:
            delete_video_file(video_file)
        except Exception as e:
            print(f'動画削除失敗: {e}')
        
        # 5秒ウェイト
        time.sleep(5)
    
    if skipped_segments > 0:
        print(f'\n警告: {skipped_segments}区間スキップ')
    
    return all_rallies


# 全区間分析実行
print(f'全区間分析開始: {len(segments)}区間')
all_rallies = analyze_all_segments(segments, player_photos, SYSTEM_PROMPT, MATCH_INFO)
print(f'\n全区間分析完了: {len(all_rallies)}ラリー検出')

In [ ]:
## セル11: 重複排除

# 重複排除実行
print('重複排除開始...')
deduped_rallies = remove_duplicates(all_rallies, threshold_sec=5.0)
print(f'重複排除完了: {len(all_rallies)} → {len(deduped_rallies)}ラリー')

In [ ]:
## セル12: 結果表示

print('=== Step 2〜3 完了 ===')
print(f'重複排除後ラリー数: {len(deduped_rallies)}')

for i, rally in enumerate(deduped_rallies, 1):
    print(f"\n--- ラリー {i} ---")
    print(f"  絶対タイムスタンプ: {rally['absolute_start_sec']:.1f}s 〜 {rally['absolute_end_sec']:.1f}s")
    print(f"  得点チーム: {rally['point_team']}")
    print(f"  決定チーム: {rally['deciding_team']}")
    print(f"  プレー種別: {rally['play_type']}")
    print(f"  確信度: {rally['confidence']:.2f}")

In [ ]:
## セル13: 一時ファイル削除

print('一時ファイル削除開始...')

# 720p再エンコード動画
try:
    Path(ENCODED_VIDEO_PATH).unlink()
    print(f'削除: {ENCODED_VIDEO_PATH}')
except Exception as e:
    print(f'削除失敗: {ENCODED_VIDEO_PATH} - {e}')

# 区間動画
for seg in segments:
    try:
        Path(seg['path']).unlink()
        print(f'削除: {seg["path"]}')
    except Exception as e:
        print(f'削除失敗: {seg["path"]} - {e}')

print('一時ファイル削除完了')

## セル13: 統合・採番・派生値・2行展開・サービスエース・整合性チェック

In [ ]:
# ============================================
# 統合・採番・派生値・2行展開・サービスエース・整合性チェック
# ============================================

def process_rallies(deduped_rallies, opponent, set_number, initial_serve_team='', initial_rotation=1):
    """ラリー処理: 採番、派生値、2行展開、サービスエース、整合性チェック"""
    
    # "相手"→実際の相手チーム名に置換
    for rally in deduped_rallies:
        if rally['point_team'] == '相手':
            rally['point_team'] = opponent
        if rally['deciding_team'] == '相手':
            rally['deciding_team'] = opponent
    
    # rally_key採番（設計書§5）
    processed_rallies = []
    rally_seq = 1
    for rally in deduped_rallies:
        rally_key = f"{MATCH_ID}_set{set_number}_rally{rally_seq:03d}"
        rally['rally_key'] = rally_key
        rally['rally_seq'] = rally_seq
        rally['set'] = set_number
        rally['match_id'] = MATCH_ID
        rally['analysis_run_id'] = ANALYSIS_RUN_ID
        rally['prompt_version'] = PROMPT_VERSION
        rally['status'] = 'HIGH' if rally['confidence'] >= 0.8 else ('MEDIUM' if rally['confidence'] >= 0.5 else 'LOW')
        rally_seq += 1
        processed_rallies.append(rally)
    
    # 派生値計算（設計書§7）
    score_us = 0
    score_them = 0
    serve_team = initial_serve_team
    rotation = initial_rotation
    
    for rally in processed_rallies:
        # スコア更新
        if rally['point_team'] == '自チーム':
            score_us += 1
        else:
            score_them += 1
        
        rally['score_us'] = score_us
        rally['score_them'] = score_them
        
        # serve_team/rotation更新
        if serve_team:
            if rally['point_team'] == '自チーム' and serve_team != '自チーム':
                serve_team = '自チーム'
                rotation = (rotation % 6) + 1
            elif rally['point_team'] == opponent and serve_team == '自チーム':
                serve_team = opponent
        
        rally['serve_team'] = serve_team if serve_team else ''
        rally['rotation'] = rotation if serve_team else ''
        
        # team/result決定（設計書§6.1.3, §7.3）
        deciding_team = rally['deciding_team']
        play_type = rally['play_type']
        
        # 2行記録判定
        is_two_line = (rally['point_team'] == opponent and 
                      rally['deciding_team'] == opponent and 
                      play_type in ['スパイク', 'フェイント', 'プッシュ'])
        rally['is_two_line'] = 'TRUE' if is_two_line else 'FALSE'
        
        # サービスエース判定（設計書§7.4）
        if rally['receive_grade'] == 'D' and rally['point_team'] == opponent:
            rally['play_type'] = 'レセプ'
            rally['result_detail'] = 'サービスエース'
            rally['is_two_line'] = 'FALSE'
        
        # team/result決定
        if is_two_line:
            rally['team'] = '自チーム'
            rally['result'] = 'ミス'
        else:
            if deciding_team == '自チーム':
                if rally['point_team'] == '自チーム':
                    rally['team'] = '自チーム'
                    rally['result'] = '得点'
                else:
                    rally['team'] = '自チーム'
                    rally['result'] = 'ミス'
            else:
                if rally['point_team'] == opponent:
                    rally['team'] = opponent
                    rally['result'] = '得点'
                else:
                    rally['team'] = opponent
                    rally['result'] = 'ミス'
        
        rally['line_index'] = 1
        
        # 2行記録展開
        if is_two_line:
            rally2 = rally.copy()
            rally2['line_index'] = 2
            rally2['team'] = opponent
            rally2['result'] = '得点'
            rally2['player'] = ''  # 初版はnull
            rally2['receiver'] = ''
            processed_rallies.append(rally2)
    
    # 整合性チェック（設計書§7.5）
    total_rallies = len([r for r in processed_rallies if r['line_index'] == 1])
    score_sum = score_us + score_them
    
    print(f'\n=== 整合性チェック ===')
    print(f'ラリー数: {total_rallies}')
    print(f'スコア合計: {score_sum}')
    print(f'最終スコア: 自チーム {score_us} - {score_them} 相手')
    
    if score_sum != total_rallies:
        print(f'警告: スコア合計({score_sum}) != ラリー数({total_rallies})')
    
    if score_us < 0 or score_them < 0:
        print(f'警告: スコアが負の値です')
    
    # セット終了スコアチェック（25点2点差）
    if score_us >= 25 or score_them >= 25:
        if abs(score_us - score_them) < 2:
            print(f'警告: セット終了スコアの2点差ルール違反')
    
    return processed_rallies


# ラリー処理実行
print('ラリー処理開始...')
opponent = MATCH_INFO.get("opponent", "相手")
set_number = MATCH_INFO.get("set", 1)
initial_serve_team = MATCH_INFO.get("serve_team", "")
initial_rotation = MATCH_INFO.get("rotation", 1)

processed_rallies = process_rallies(deduped_rallies, opponent, set_number, initial_serve_team, initial_rotation)
print(f'ラリー処理完了: {len(deduped_rallies)} → {len(processed_rallies)}行（2行展開含む）')

## セル14: Google Sheets認証・シート書き込み

In [ ]:
# ============================================
# Google Sheets認証・シート書き込み
# ============================================

import gspread
from google.auth import default
from google.oauth2.credentials import Credentials

# Google Sheets認証
print('Google Sheets認証中...')
if IN_COLAB:
    # Colabの場合はデフォルト認証
    creds, _ = default()
    gc = gspread.authorize(creds)
else:
    # ローカルの場合はサービスアカウント認証
    SERVICE_ACCOUNT_FILE = get_secret('GOOGLE_SERVICE_ACCOUNT_FILE')
    if not SERVICE_ACCOUNT_FILE:
        raise ValueError('GOOGLE_SERVICE_ACCOUNT_FILE が設定されていません')
    gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)

print('認証完了')

# スプレッドシートID（ユーザー入力）
SPREADSHEET_ID = input("スプレッドシートID: ").strip()
if not SPREADSHEET_ID:
    raise ValueError('スプレッドシートIDが入力されていません')

# スプレッドシートを開く
sh = gc.open_by_key(SPREADSHEET_ID)
print(f'スプレッドシートを開きました: {sh.title}')

# AI提案シートの処理
AI_SHEET_NAME = "AI提案"

# 再実行時削除（設計書§11: 同一match_id+setでstatus!=COMMITTEDの行を削除）
try:
    ai_sheet = sh.worksheet(AI_SHEET_NAME)
    print(f'{AI_SHEET_NAME}シートが存在します')
    
    # 既存データを取得
    existing_data = ai_sheet.get_all_values()
    if len(existing_data) > 1:
        # ヘッダー行をスキップしてデータを確認
        header = existing_data[0]
        # match_id列とset列のインデックスを探す
        match_id_idx = header.index('match_id') if 'match_id' in header else -1
        set_idx = header.index('set') if 'set' in header else -1
        status_idx = header.index('status') if 'status' in header else -1
        
        if match_id_idx >= 0 and set_idx >= 0 and status_idx >= 0:
            rows_to_delete = []
            for i, row in enumerate(existing_data[1:], start=2):  # 1-indexed
                if len(row) > max(match_id_idx, set_idx, status_idx):
                    row_match_id = row[match_id_idx] if match_id_idx < len(row) else ''
                    row_set = row[set_idx] if set_idx < len(row) else ''
                    row_status = row[status_idx] if status_idx < len(row) else ''
                    
                    if row_match_id == MATCH_ID and str(row_set) == str(set_number) and row_status != 'COMMITTED':
                        rows_to_delete.append(i)
            
            # 後ろから削除してインデックスズレを防ぐ
            rows_to_delete.sort(reverse=True)
            for row_idx in rows_to_delete:
                ai_sheet.delete_rows(row_idx)
            
            if rows_to_delete:
                print(f'再実行時削除: {len(rows_to_delete)}行削除')
except gspread.WorksheetNotFound:
    print(f'{AI_SHEET_NAME}シートが存在しません。新規作成します')
    ai_sheet = sh.add_worksheet(title=AI_SHEET_NAME, rows=1000, cols=43)

# ヘッダー行定義（設計書§5.2）
AI_HEADERS = [
    'rally_key', 'line_index', 'is_two_line', 'rally_seq', 'rally_start_sec', 'rally_end_sec',
    'confidence', 'status', 'date', 'opponent', 'set', 'score_us', 'score_them',
    'point_team', 'serve_team', 'rotation', 'deciding_team', 'receive_grade', 'receiver',
    'team', 'player', 'play_type', 'result', 'result_detail', 'attack_type',
    'blocker_count', 'zone_from', 'zone_to', 'note', 'ai_note', 'field_confidences',
    'original_payload', 'final_payload', 'human_modified', 'match_id', 'analysis_run_id',
    'prompt_version', 'approved_at', 'initial_serve_team', 'initial_rotation', 'approved_by'
]

# ヘッダー行を設定（シートが空の場合のみ）
if ai_sheet.row_count == 1 and len(ai_sheet.row_values(1)) == 0:
    ai_sheet.append_row(AI_HEADERS)
    print('ヘッダー行を設定しました')

# データ行を構築
def build_canonical_json(rally):
    """canonical JSONを構築（設計書§8.2）"""
    comparison_fields = ['point_team', 'deciding_team', 'receive_grade', 'receiver', 
                        'play_type', 'player', 'result_detail', 'attack_type',
                        'blocker_count', 'zone_from', 'zone_to', 'our_defense_type']
    canonical = {}
    for field in comparison_fields:
        canonical[field] = rally.get(field, '')
    return json.dumps(canonical, sort_keys=True, ensure_ascii=False)

rows_to_write = []
for rally in processed_rallies:
    row = [
        rally.get('rally_key', ''),
        rally.get('line_index', 1),
        rally.get('is_two_line', 'FALSE'),
        rally.get('rally_seq', 0),
        rally.get('rally_start_sec', 0),
        rally.get('rally_end_sec', 0),
        rally.get('confidence', 0),
        rally.get('status', ''),
        rally.get('date', MATCH_DATE),
        rally.get('opponent', opponent),
        rally.get('set', set_number),
        rally.get('score_us', 0),
        rally.get('score_them', 0),
        rally.get('point_team', ''),
        rally.get('serve_team', ''),
        rally.get('rotation', ''),
        rally.get('deciding_team', ''),
        rally.get('receive_grade', ''),
        rally.get('receiver', ''),
        rally.get('team', ''),
        rally.get('player', ''),
        rally.get('play_type', ''),
        rally.get('result', ''),
        rally.get('result_detail', ''),
        rally.get('attack_type', ''),
        rally.get('blocker_count', ''),
        rally.get('zone_from', ''),
        rally.get('zone_to', ''),
        rally.get('note', ''),
        rally.get('ai_note', ''),
        json.dumps(rally.get('field_confidences', {}), ensure_ascii=False),
        build_canonical_json(rally) if rally.get('line_index') == 1 else '',  # line1のみ
        '',  # final_payloadは空
        '',  # human_modifiedは空
        rally.get('match_id', MATCH_ID),
        rally.get('analysis_run_id', ANALYSIS_RUN_ID),
        rally.get('prompt_version', PROMPT_VERSION),
        '',  # approved_atは空
        initial_serve_team,
        initial_rotation,
        ''  # approved_byは空
    ]
    rows_to_write.append(row)

# データを書き込み
if rows_to_write:
    ai_sheet.append_rows(rows_to_write)
    print(f'データ書き込み完了: {len(rows_to_write)}行')

# サマリー作成
high_count = len([r for r in processed_rallies if r['status'] == 'HIGH'])
medium_count = len([r for r in processed_rallies if r['status'] == 'MEDIUM'])
low_count = len([r for r in processed_rallies if r['status'] == 'LOW'])

print(f'\n=== サマリー ===')
print(f'総ラリー数: {len([r for r in processed_rallies if r["line_index"] == 1])}')
print(f'HIGH: {high_count}')
print(f'MEDIUM: {medium_count}')
print(f'LOW: {low_count}')
print(f'確信度平均: {sum(r["confidence"] for r in processed_rallies if r["line_index"] == 1) / len([r for r in processed_rallies if r["line_index"] == 1]):.2f}')

## セル15: DataFrame表示・確信度分布・エラー箇所

In [ ]:
# ============================================
# DataFrame表示・確信度分布・エラー箇所
# ============================================

import pandas as pd

# DataFrame作成（line_index=1のみ）
df_rallies = pd.DataFrame([r for r in processed_rallies if r['line_index'] == 1])

print('=== DataFrame表示 ===')
display_cols = ['rally_seq', 'point_team', 'deciding_team', 'play_type', 'confidence', 'status', 'team', 'result']
print(df_rallies[display_cols].head(10))

print('\n=== 確信度分布 ===')
print(f'平均確信度: {df_rallies["confidence"].mean():.3f}')
print(f'最小確信度: {df_rallies["confidence"].min():.3f}')
print(f'最大確信度: {df_rallies["confidence"].max():.3f}')

# 確信度ヒストグラム
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist(df_rallies['confidence'], bins=20, edgecolor='black')
plt.xlabel('確信度')
plt.ylabel('ラリー数')
plt.title('確信度分布')
plt.show()

# ステータス分布
status_counts = df_rallies['status'].value_counts()
print(f'\nステータス分布:')
print(status_counts)

# エラー箇所（確信度が低いラリー）
low_confidence = df_rallies[df_rallies['confidence'] < 0.5]
if not low_confidence.empty:
    print(f'\n=== 低確信度ラリー（確信度 < 0.5） ===')
    print(low_confidence[display_cols])
else:
    print('\n低確信度ラリーはありません')